# AstroShield — Evolucionar una constelación inteligente
**Proyecto en equipo - Ingeniería en Tecnologías de la Información**

Este notebook contiene la implementación completa del sistema de optimización de una constelación de 8 satélites utilizando un algoritmo evolutivo, restricciones espaciales y visualización 3D interactiva en Plotly.

In [3]:
import math
import random
import numpy as np
import pandas as pd
import plotly.graph_objects as go

ModuleNotFoundError: No module named 'pandas'

## Sección 1: Matemáticas, Distancias y Restricciones (Integrante 1)
Aquí se implementa la fórmula de Haversine, el cálculo de cobertura de las 6 regiones y la validación de las zonas de exclusión (desechos y objetos ocupados).

In [ ]:
# Datos iniciales de las regiones (Latitud, Longitud, Prioridad: 3=Alta, 2=Media, 1=Normal)
REGIONS = {
    "Yucatán": {"coords": (20.5, -89.0), "priority": 3},
    "Amazonía": {"coords": (-3.5, -62.0), "priority": 3},
    "California": {"coords": (36.8, -119.4), "priority": 2},
    "Sahel": {"coords": (15.0, 0.0), "priority": 2},
    "Japón": {"coords": (36.0, 138.0), "priority": 2},
    "Australia": {"coords": (-25.0, 133.0), "priority": 1}
}

# Zonas restringidas por desechos (Centro lat, lon, radio en grados)
DEBRIS_ZONES = [
    {"center": (45.0, -35.0), "radius": 12.0},
    {"center": (-30.0, -120.0), "radius": 10.0},
    {"center": (78.0, 30.0), "radius": 8.0}
]

# Objetos ocupados existentes (Centro lat, lon, radio de exclusión)
BLOCKED_OBJECTS = [
    {"center": (10.0, 45.0), "radius": 5.0},
    {"center": (-15.0, 110.0), "radius": 5.0}
]

def haversine(lat1, lon1, lat2, lon2):
    """Calcula la distancia en kilómetros entre dos puntos sobre la Tierra."""
    R = 6371.0  # Radio de la Tierra en km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon / 2)**2
    c = 2 * math.asin(math.sqrt(a))
    return R * c

def is_valid_position(lat, lon):
    """Verifica si una posición viola las zonas de desechos o los objetos ocupados."""
    # Verificar desechos espaciales
    for zone in DEBRIS_ZONES:
        dist = haversine(lat, lon, zone["center"][0], zone["center"][1])
        # Aproximación de grados a km (~111 km por grado)
        if dist <= zone["radius"] * 111:
            return False
            
    # Verificar objetos ocupados
    for obj in BLOCKED_OBJECTS:
        dist = haversine(lat, lon, obj["center"][0], obj["center"][1])
        if dist <= obj["radius"] * 111:
            return False
            
    return True

print("Módulo de matemáticas y restricciones cargado.")

## Sección 2: Algoritmo Evolutivo (Integrante 2)
Implementación de la población de 8 satélites, operadores genéticos (selección, cruce, mutación) y la función de fitness.

In [4]:
import random
import numpy as np

class AlgoritmoEvolutivoConstelacion:
    def __init__(self, evaluar_fitness_func, validar_posicion_func=None, num_satelites=8):
        """
        Inicializa el algoritmo evolutivo para la constelación AstroShield.
        """
        self.evaluar_fitness = evaluar_fitness_func
        self.validar_posicion = validar_posicion_func
        self.num_satelites = num_satelites

    def generar_individuo(self):
        """
        Genera un individuo válido (8 satélites con coordenadas [lat, lon]).
        """
        individuo = []
        intentos = 0
        while len(individuo) < self.num_satelites and intentos < 1000:
            lat = random.uniform(-80, 80)
            lon = random.uniform(-180, 180)
            
            if self.validar_posicion is None or self.validar_posicion(lat, lon):
                individuo.append([lat, lon])
            intentos += 1
            
        while len(individuo) < self.num_satelites:
            individuo.append([random.uniform(-80, 80), random.uniform(-180, 180)])
            
        return individuo

    def generar_poblacion_inicial(self, tamano_poblacion):
        """Genera la población inicial de configuraciones."""
        return [self.generar_individuo() for _ in range(tamano_poblacion)]

    def seleccion_torneo(self, poblacion, aptitudes, k=3):
        """Selecciona al mejor individuo mediante torneo."""
        seleccionados = random.sample(list(zip(poblacion, aptitudes)), k)
        seleccionados.sort(key=lambda x: x[1], reverse=True)
        return seleccionados[0][0]

    def cruce(self, padre1, padre2):
        """Operador de cruce de un punto entre dos configuraciones."""
        punto = random.randint(1, self.num_satelites - 1)
        hijo1 = padre1[:punto] + padre2[punto:]
        hijo2 = padre2[:punto] + padre1[punto:]
        return hijo1, hijo2

    def mutacion(self, individuo, tasa_mutacion=0.1):
        """Aplica una ligera perturbación aleatoria a las coordenadas de los satélites."""
        individuo_mutado = []
        for sat in individuo:
            if random.random() < tasa_mutacion:
                lat = max(-90, min(90, sat[0] + random.gauss(0, 3)))
                lon = max(-180, min(180, sat[1] + random.gauss(0, 3)))
                individuo_mutado.append([lat, lon])
            else:
                individuo_mutado.append(sat)
        return individuo_mutado

    def ejecutar(self, generaciones=50, tamano_poblacion=30, tasa_mutacion=0.1):
        """
        Ejecuta el ciclo principal del algoritmo evolutivo y registra el historial.
        """
        poblacion = self.generar_poblacion_inicial(tamano_poblacion)
        historial_max = []
        historial_avg = []
        mejor_individuo = None
        mejor_fitness = float('-inf')

        for gen in range(generaciones):
            aptitudes = [self.evaluar_fitness(ind) for ind in poblacion]
            
            max_gen = max(aptitudes)
            avg_gen = sum(aptitudes) / len(aptitudes)
            historial_max.append(max_gen)
            historial_avg.append(avg_gen)

            idx_mejor = np.argmax(aptitudes)
            if aptitudes[idx_mejor] > mejor_fitness:
                mejor_fitness = aptitudes[idx_mejor]
                mejor_individuo = poblacion[idx_mejor]

            nueva_poblacion = []
            while len(nueva_poblacion) < tamano_poblacion:
                p1 = self.seleccion_torneo(poblacion, aptitudes)
                p2 = self.seleccion_torneo(poblacion, aptitudes)
                h1, h2 = self.cruce(p1, p2)
                
                nueva_poblacion.append(self.mutacion(h1, tasa_mutacion))
                if len(nueva_poblacion) < tamano_poblacion:
                    nueva_poblacion.append(self.mutacion(h2, tasa_mutacion))

            poblacion = nueva_poblacion

        return mejor_individuo, mejor_fitness, historial_max, historial_avg

## Sección 3: Visualización 3D Interactiva (Integrante 3)
Generación del globo terráqueo interactivo usando Plotly para mostrar satélites, regiones y zonas de exclusión.

## Sección 4: Pruebas y Validación (Integrante 4)
Ejecución de pruebas iniciales, verificación de restricciones y análisis de múltiples corridas.

## Sección 5: Análisis de Resultados y Conclusiones (Integrante 5)
> **Definición del problema y análisis técnico:**
> * Las restricciones espaciales se aplicaron mediante penalizaciones matemáticas en la función de fitness.
> * El algoritmo logra explorar configuraciones válidas evitando los puntos prohibidos de desechos espaciales.